# Load Products JSON into SQLite Database

This notebook reads `all_products_info.json` and stores every product into a SQLite database (`products.db`) inside a `products` table.

In [1]:
import json
import sqlite3
from pathlib import Path

# File paths
JSON_FILE = Path('all_products_info.json')
DB_FILE = Path('products.db')

# Load the JSON data
with open(JSON_FILE, 'r', encoding='utf-8') as f:
    data = json.load(f)

print(f'Loaded {len(data)} categories from {JSON_FILE.name}')

Loaded 21 categories from all_products_info.json


In [2]:
# Connect to SQLite (creates the file if it doesn't exist)
conn = sqlite3.connect(DB_FILE)
cursor = conn.cursor()

# Drop the table if it exists so we can start fresh
cursor.execute('DROP TABLE IF EXISTS products')

# Create the products table
cursor.execute('''
    CREATE TABLE products (
        id       INTEGER PRIMARY KEY AUTOINCREMENT,
        category TEXT    NOT NULL,
        title    TEXT    NOT NULL,
        rating   TEXT,
        price    TEXT,
        link     TEXT
    )
''')

conn.commit()
print('Table "products" created successfully.')

Table "products" created successfully.


In [ ]:
# Flatten the JSON into a list of tuples for bulk insert
rows = []
for category, products in data.items():
    for product in products:
        rows.append((
            category,
            product.get('title', ''),
            product.get('rating', ''),
            product.get('price', ''),
            product.get('link', ''),
        ))

# Bulk insert all rows
cursor.executemany(
    'INSERT INTO products (category, title, rating, price, link) VALUES (?, ?, ?, ?, ?)',
    rows,
)
conn.commit()

print(f'Inserted {len(rows)} products into the database.')

Inserted 403 products into the database.


In [4]:
# Verify: total rows
cursor.execute('SELECT COUNT(*) FROM products')
print('Total products in DB:', cursor.fetchone()[0])

Total products in DB: 403


In [5]:
# Verify: counts per category
cursor.execute('SELECT category, COUNT(*) FROM products GROUP BY category ORDER BY COUNT(*) DESC')
print('\nProducts per category:')
for category, count in cursor.fetchall():
    print(f'  {category:35s} {count}')


Products per category:
  Guitars                             198
  Digital Pianos                      49
  Violins                             29
  Keyboard Instruments                23
  Drums                               18
  Synthesizers                        17
  Studio Monitor                      16
  Microphones                         11
  Audio Streaming                     9
  Audio Interface                     7
  Loudspeaker                         6
  Portable PA System                  5
  Mixing Console                      4
  Portable USB & Bluetooth Speakerphone 3
  Sound Bars                          2
  Game Streaming                      2
  Recorders                           1
  Pianica                             1
  Headphones & Earphones              1
  Headphones                          1


In [6]:
# Verify: preview first 5 rows
cursor.execute('SELECT id, category, title, rating, price FROM products LIMIT 5')
print('\nSample rows:')
for row in cursor.fetchall():
    print(row)


Sample rows:
(1, 'Keyboard Instruments', 'Yamaha PSR-SX720+ Arranger workstation with 61 Keys', '', 'MRP₹ 121,990')
(2, 'Keyboard Instruments', 'Yamaha PSR-I510 Portable Keyboard', '', 'MRP₹ 27,990')
(3, 'Keyboard Instruments', 'Yamaha PA-130B AC Power Adaptor', '5', 'MRP₹ 1,840')
(4, 'Keyboard Instruments', 'Yamaha PSS-E30 Mini Keyboard For Kids (Made in India)', '5', 'MRP₹ 5,195')
(5, 'Keyboard Instruments', 'Yamaha PSS-F30 Mini Keyboard For Kids (Made in India)', '4.9', 'MRP₹ 5,195')


In [7]:
# Close the database connection
conn.close()
print('Database connection closed.')

Database connection closed.
